In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
def fractional_release(t, D, R, n_terms=50):
    """
    Compute M_t / M_inf for diffusion out of a sphere.

    Parameters
    ----------
    t : float or array_like
        Time(s) at which to evaluate the release. Same time units as implied by D.
    D : float
        Diffusion coefficient (e.g. m^2/s). Must be > 0.
    R : float
        Sphere radius (same length units as D, e.g. m). Must be > 0.
    n_terms : int, optional
        Number of terms in the series (default 50 — gives machine precision
        for any t > 0 because of the exponential decay in n^2).

    Returns
    -------
    Mt_over_Minf : ndarray
        Fractional mass released, in [0, 1]. Same shape as `t`.
    """
    if D <= 0 or R <= 0:
        raise ValueError("D and R must be positive.")
    if n_terms < 1:
        raise ValueError("n_terms must be >= 1.")

    t = np.asarray(t, dtype=float)

    # n = 1, 2, ..., n_terms  -> shape (n_terms,)
    n = np.arange(1, n_terms + 1)

    # Broadcast: t has shape (...,), n has shape (n_terms,)
    # exponent has shape (..., n_terms)
    exponent = -D * (n**2) * (np.pi**2) * t[..., np.newaxis] / (R**2)
    series = np.sum(np.exp(exponent) / n**2, axis=-1)

    result = 1.0 - (6.0 / np.pi**2) * series

    # At t = 0 the series sums to pi^2/6 exactly, giving 0.0 — but clip tiny
    # negative values from floating-point error.
    return np.clip(result, 0.0, 1.0)

In [4]:
df_profiles = pd.read_excel("plga_dataset/mp_dataset_processed.xlsx")
df_summary = pd.read_excel("plga_dataset/crank_fit.xlsx")

In [ ]:
df_release_new = pd.DataFrame()
df_release_new_even = pd.DataFrame()

number_of_samples = 50 # how many points in uniform release
indices = np.linspace(0, 1, number_of_samples) # linear indics of points in uniform release

for i in range(len(df_profiles['Formulation Index'].unique())):

    selected_index = i + 1
    df_profile_0 = df_profiles[df_profiles['Formulation Index'] == selected_index].reset_index(drop=True)
    df_profile_0 = df_profile_0[df_profile_0['Release'] >= 0].reset_index(drop=True)
    df_profile_0 = df_profile_0[df_profile_0['Release'] <= 1].reset_index(drop=True)

    df_summary_0 = df_summary[df_summary['Formulation Index'] == selected_index].reset_index(drop=True)    
    particle_radius = df_summary_0['Particle Radius'].iloc[0] * 10**-6 # micrometers
    D = df_summary_0['Crank_D'].iloc[0] # m^2/sec

    df_profile_0['Crank_D'] = D
    
    time_array = df_profile_0['Time'] * 24 * 3600 # sec


    ## START - Construct new df with same time samplign
    max_release_time = time_array.max() # seconds
    # even_release_time_array = max_release_time * (np.exp(indices) - 1) / (np.exp(1) - 1) # Exponential distribution
    even_release_time_array = max_release_time * (indices ** 1.5) # Power law distribution
    even_release_fraction_array = np.interp(even_release_time_array, time_array, df_profile_0['Release'])

    df_profile_0_even = pd.DataFrame({
        "Time" : even_release_time_array / (24*3600),
        "Release" : even_release_fraction_array
    })

    df_profile_0_even['Crank_D'] = D

    for col in df_profile_0.drop(columns=['Time', 'Release', 'Crank_D']).columns:
        df_profile_0_even[col] = df_profile_0[col].iloc[0]

    ## END - Construct new df with same time samplign


    ## START - Get Crank release at even timestamps
    diffusion_release = fractional_release(even_release_time_array, D, particle_radius, n_terms=170)

    df_profile_0_even['Crank_Release'] = diffusion_release
    ## END - Get Crank release at even timestamps

    df_release_new_even = pd.concat([df_release_new_even, df_profile_0_even])
    
    plt.scatter(df_profile_0['Time'], df_profile_0['Release'], marker='x', label='Original')
    plt.plot(df_profile_0_even['Time'], df_profile_0_even['Release'], color='orange', label='Original')
    plt.plot(df_profile_0_even['Time'], df_profile_0_even['Crank_Release'], color='green', label='Crank')
    plt.legend()
    plt.grid()
    plt.show()
    
    # break

df_release_new_even.reset_index(drop=True, inplace=True)

In [6]:
df_release_new_even

,Time,Release,Crank_D,Formulation Index,Drug MW,Drug TPSA,Drug LogP,Polymer MW,LA/GA,Initial Drug-to-Polymer Ratio,Particle Size,Drug Loading Capacity,Drug Encapsulation Efficiency,Solubility Enhancer Concentration,Crank_Release
0,0.000000,0.000000,7.085839e-19,1,639.830,116.04,5.7289,75.0,3.0,0.666667,47.723,35.41,88.30,0.5,0.003566
1,0.693097,0.022968,7.085839e-19,1,639.830,116.04,5.7289,75.0,3.0,0.666667,47.723,35.41,88.30,0.5,0.029000
2,1.960374,0.062544,7.085839e-19,1,639.830,116.04,5.7289,75.0,3.0,0.666667,47.723,35.41,88.30,0.5,0.048515
3,3.601438,0.060723,7.085839e-19,1,639.830,116.04,5.7289,75.0,3.0,0.666667,47.723,35.41,88.30,0.5,0.065453
4,5.544776,0.058567,7.085839e-19,1,639.830,116.04,5.7289,75.0,3.0,0.666667,47.723,35.41,88.30,0.5,0.080867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16045,24.666230,0.918572,2.293913e-17,321,360.406,84.12,1.1031,20.0,3.0,0.112347,42.360,7.96,80.16,0.0,0.790574
16046,25.492988,0.920141,2.293913e-17,321,360.406,84.12,1.1031,20.0,3.0,0.112347,42.360,7.96,80.16,0.0,0.798194
16047,26.328783,0.921728,2.293913e-17,321,360.406,84.12,1.1031,20.0,3.0,0.112347,42.360,7.96,80.16,0.0,0.805595
16048,27.173517,0.923332,2.293913e-17,321,360.406,84.12,1.1031,20.0,3.0,0.112347,42.360,7.96,80.16,0.0,0.812781


In [7]:
df_release_new_even.to_excel("plga_dataset/release_dataset_even.xlsx", index=False)